# Machine Learning CIE 2 – Activity 2
## Simple House Price Prediction using Linear Regression

**Unit:** Unit II – Supervised Learning: Regression  
**Dataset:** California Housing Dataset  
**Source:** [Scikit-learn California Housing documentation](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_california_housing.html)  
**Input (X):** `MedInc` – median income in tens of thousands of US dollars.  
**Target (y):** `HousePriceUSD` – median house value converted to US dollars.

This notebook uses one input feature to build a Simple Linear Regression model. It does not use record IDs as features.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 2. Load and inspect the dataset

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

# Original MedHouseVal is in units of $100,000; convert it to US dollars.
df["HousePriceUSD"] = df["MedHouseVal"] * 100000

print("Dataset: California Housing Dataset")
print("Source: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_california_housing.html")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\\nFirst five rows:")
display(df.head())
print("\\nLast five rows:")
display(df.tail())
print("\\nColumn names:", df.columns.tolist())
print("\\nData types:")
print(df.dtypes)
print("\\nDataset information:")
df.info()

### Dataset column description
- `MedInc`: Median income in the block group (tens of thousands of US dollars).
- `HouseAge`: Median house age in years.
- `AveRooms`: Average rooms per household.
- `AveBedrms`: Average bedrooms per household.
- `Population`: Block-group population.
- `AveOccup`: Average household occupancy.
- `Latitude`, `Longitude`: Geographic location.
- `MedHouseVal`: Original median house value in units of $100,000.
- `HousePriceUSD`: Converted median house value in US dollars.

The target is continuous and numerical.

## 3. Summary statistics and data-quality checks

In [ ]:
# Remove exact duplicate full records only. Repeated MedInc/price pairs alone
# are not automatically removed because they may be genuine different observations.
df_checked = df.drop_duplicates().copy()

model_data = df_checked[["MedInc", "HousePriceUSD"]].copy()
model_data = model_data.replace([np.inf, -np.inf], np.nan)
model_data = model_data.dropna(subset=["MedInc", "HousePriceUSD"])
model_data = model_data[
    (model_data["MedInc"] >= 0) &
    (model_data["HousePriceUSD"] > 0)
].copy()

print("Rows in original dataset:", len(df))
print("Exact duplicate full rows removed:", len(df) - len(df_checked))
print("Rows after missing/invalid-value checks:", len(model_data))
print("Rows available for modeling:", len(model_data))
print("Remaining missing values:", model_data.isnull().sum().sum())

### Cleaning explanation
The selected feature and target are checked for missing and invalid values. Exact duplicate selected-feature/target records are removed if present. The original dataset is preserved in `original_df`. Review the printed results and report the values actually shown.

## 4. Select input feature (X) and target (y)

In [ ]:
X = model_data[["MedInc"]]
y = model_data["HousePriceUSD"]

print("Input feature X: MedInc")
print("Target y: HousePriceUSD")
display(X.head())
display(y.head())

### Why choose MedInc?
Median income may be related to house values in an area, so it is used as the input feature. HousePriceUSD is the continuous numerical target. An apparent relationship does not prove causation.

## 5. Visualize the relationship

In [ ]:
plt.figure(figsize=(9, 6))
plt.scatter(X["MedInc"], y, alpha=0.25)
plt.title("Median Income vs House Price")
plt.xlabel("Median Income (tens of thousands of US dollars)")
plt.ylabel("Median House Price (US dollars)")
plt.tight_layout()
plt.show()

**Observation:** Inspect the scatter plot and describe whether the overall relationship appears positive, negative, weak or non-linear. A visible relationship does not by itself prove causation.

## 6. Train-test split (80:20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))
print("Total observations:", len(X_train) + len(X_test))

Training data is used to learn model parameters. Test data is kept separate and used to evaluate performance on unseen observations.

## 7. Train Simple Linear Regression

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

coefficient = model.coef_[0]
intercept = model.intercept_

print("Coefficient (slope):", coefficient)
print("Intercept:", intercept)
print(f"Regression equation: HousePriceUSD = {intercept:.2f} + ({coefficient:.2f} * MedInc)")

### Coefficient and intercept
- **Slope/coefficient:** Estimated change in house price (USD) for a one-unit increase in MedInc. One unit of MedInc represents $10,000 of median income.
- **Intercept:** Estimated house price when MedInc is zero; it is a mathematical starting point and may not be meaningful in every real-world context.

## 8. Generate test-set predictions

In [ ]:
y_pred = model.predict(X_test)
comparison = pd.DataFrame({
    "Actual House Price (USD)": y_test.values,
    "Predicted House Price (USD)": y_pred
})
display(comparison.head(10))

Compare actual and predicted test values. Differences are prediction errors. These are test-set results, not training predictions.

## 9. Evaluate the model

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MAE  : ${mae:,.2f}")
print(f"MSE  : {mse:,.2f}")
print(f"RMSE : ${rmse:,.2f}")
print(f"R2   : {r2:.4f}")

### Metric interpretation
- **MAE:** Average absolute prediction error, in US dollars.
- **MSE:** Average squared prediction error; large errors receive greater penalty, and its unit is squared dollars.
- **RMSE:** Square root of MSE; it is in the target's original units (US dollars).
- **R²:** Compares the model with a mean-based baseline. Closer to 1 generally indicates a better fit; test R² can be negative.

Interpret the values printed above in relation to the price scale; do not report invented results.

## 10. Actual vs predicted visualization

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.3, label="Test predictions")
low = min(y_test.min(), y_pred.min())
high = max(y_test.max(), y_pred.max())
plt.plot([low, high], [low, high], "r--", label="Perfect prediction line")
plt.title("Actual vs Predicted House Prices")
plt.xlabel("Actual House Price (USD)")
plt.ylabel("Predicted House Price (USD)")
plt.legend()
plt.tight_layout()
plt.show()

**Observation:** Points close to the dashed reference line indicate more accurate predictions. Points farther away indicate larger errors. Describe what the graph shows after running it.

## 11. Predict a new example

In [ ]:
# Use the median training income, which is within the training data range.
new_income = float(X_train["MedInc"].median())
new_price = model.predict(pd.DataFrame({"MedInc": [new_income]}))[0]

print(f"New example MedInc: {new_income:.2f} (tens of thousands of USD)")
print(f"Predicted house price: ${new_price:,.2f}")

The example uses a median income value from the training data. The result is an estimate from this one-feature model, not a guaranteed market price.

## 12. Save CSV outputs

In [ ]:
model_data.to_csv("california_housing_selected_data.csv", index=False)
comparison.to_csv("test_actual_vs_predicted.csv", index=False)
print("Saved: california_housing_selected_data.csv")
print("Saved: test_actual_vs_predicted.csv")

## 13. Conclusion
In this activity, the California Housing Dataset was loaded and inspected. Data types, summary statistics, missing values and duplicate records were checked. `MedInc` was selected as the input feature and `HousePriceUSD` as the continuous target. The data was divided into 80% training and 20% testing sets. A Simple Linear Regression model was trained and used to predict test-set prices. Performance was evaluated using MAE, MSE, RMSE and R². Scatter plots were used to inspect the relationship and compare actual and predicted prices. A new example prediction was also generated.

**Limitation:** House prices depend on several factors, including location, property size, condition, age and market conditions. A model using only median income cannot capture all of them. This dataset is based on 1990 California census information and is not a source of current market prices.

## 14. Submission checklist
- [ ] Run all notebook cells and ensure outputs and graphs are visible.
- [ ] Download the notebook and generated CSV files.
- [ ] Upload the notebook, README, requirements and generated CSV files to a separate public GitHub repository.
- [ ] Check that the repository opens without access requests.
- [ ] Submit the public repository URL as instructed by faculty.